# 04 — Agent 2 : consolidation multi-échelles puis agrégation en portions

Ce notebook reçoit les annotations atomiques produites par l'Agent 1.

Nouvelle logique :

```text
annotations atomiques multi-échelles
            ↓
ÉTAPE A — consolider une même caractéristique
            ↓
annotations consolidées
            ↓
ÉTAPE B — combiner les caractéristiques différentes
            ↓
portions homogènes de l'itinéraire
```

C'est important : une `pente_douce` observée à 25 m et à 100 m
ne doit pas être traitée comme deux caractéristiques différentes.
Ce sont deux **preuves à des échelles différentes du même phénomène**.

In [ ]:
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd
from pydantic import BaseModel, Field

## 1. Charger les vraies annotations du notebook 03

Le notebook 03 doit avoir produit :

```text
data/output/agent1_annotations_atomic.json
```

In [ ]:
annotations_path = Path("data/output/agent1_annotations_atomic.json")

if not annotations_path.exists():
    raise FileNotFoundError(
        "Annotations de l'Agent 1 introuvables. "
        "Exécute d'abord le notebook 03 jusqu'à l'export "
        "data/output/agent1_annotations_atomic.json."
    )

with annotations_path.open("r", encoding="utf-8") as f:
    raw_annotations = json.load(f)

print("Annotations atomiques chargées :", len(raw_annotations))

display(pd.DataFrame([
    {
        "feature": a["feature"],
        "start_m": a["start_m"],
        "end_m": a["end_m"],
        "resolution_m": a["resolution_m"],
        "confidence": a["confidence"],
    }
    for a in raw_annotations[:20]
]))

## 2. Modèles de données

Une annotation consolidée conserve toutes les observations qui ont servi
à la construire.

On ne perd donc pas l'information :

```text
25 m
50 m
100 m
```

Au contraire, on la stocke dans `evidence`.

In [ ]:
class AtomicAnnotation(BaseModel):
    feature: str
    start_m: float
    end_m: float
    resolution_m: float
    confidence: float
    metrics: dict = Field(default_factory=dict)


class ConsolidatedAnnotation(BaseModel):
    feature: str
    start_m: float
    end_m: float

    # Échelles ayant soutenu cette annotation
    resolutions_m: list[float]

    # Nombre total d'observations atomiques fusionnées
    n_evidence: int

    # Résumés simples de confiance
    mean_confidence: float
    max_confidence: float

    # True si la zone est détectée à plusieurs échelles
    multi_scale: bool

    # Toutes les observations originales sont conservées
    evidence: list[AtomicAnnotation]


class Portion(BaseModel):
    start_m: float
    end_m: float
    annotations: list[str]

    # Une entrée détaillée par caractéristique active
    details: list[dict] = Field(default_factory=list)

In [ ]:
atomic_annotations = [
    AtomicAnnotation(**a)
    for a in raw_annotations
]

## 3. Étape A — Fusionner les détections d'une même caractéristique

Exemple :

```text
pente_douce 25 m  : 100────125
pente_douce 25 m  :    110────135
pente_douce 100 m : 80────────────────180
```

devient :

```text
pente_douce : 80──────────────────────180
résolutions observées : [25, 100]
```

La fusion se fait si les intervalles :

- se chevauchent ;
- ou sont séparés par au plus `merge_gap_m`.

Le défaut choisi est `10 m`, soit le pas spatial de base du projet.

In [ ]:
def _make_consolidated(feature, group):
    resolutions = sorted({
        float(a.resolution_m)
        for a in group
    })

    confidences = [
        float(a.confidence)
        for a in group
    ]

    return ConsolidatedAnnotation(
        feature=feature,
        start_m=min(a.start_m for a in group),
        end_m=max(a.end_m for a in group),
        resolutions_m=resolutions,
        n_evidence=len(group),
        mean_confidence=sum(confidences) / len(confidences),
        max_confidence=max(confidences),
        multi_scale=len(resolutions) > 1,
        evidence=list(group),
    )


def consolidate_same_feature(
    annotations,
    merge_gap_m=10.0,
):
    by_feature = defaultdict(list)

    for annotation in annotations:
        by_feature[annotation.feature].append(annotation)

    consolidated = []

    for feature, feature_annotations in by_feature.items():
        ordered = sorted(
            feature_annotations,
            key=lambda a: (a.start_m, a.end_m)
        )

        if not ordered:
            continue

        current_group = [ordered[0]]
        current_end = ordered[0].end_m

        for annotation in ordered[1:]:
            # Chevauchement ou très petit trou spatial.
            if annotation.start_m <= current_end + merge_gap_m:
                current_group.append(annotation)
                current_end = max(current_end, annotation.end_m)
            else:
                consolidated.append(
                    _make_consolidated(feature, current_group)
                )
                current_group = [annotation]
                current_end = annotation.end_m

        consolidated.append(
            _make_consolidated(feature, current_group)
        )

    return sorted(
        consolidated,
        key=lambda a: (a.start_m, a.end_m, a.feature)
    )

In [ ]:
consolidated_annotations = consolidate_same_feature(
    atomic_annotations,
    merge_gap_m=10.0,
)

print(
    "Avant consolidation :",
    len(atomic_annotations),
    "annotations atomiques"
)

print(
    "Après consolidation :",
    len(consolidated_annotations),
    "annotations"
)

## 4. Vérifier l'effet de la consolidation

Ce tableau permet de vérifier qu'une caractéristique détectée plusieurs fois
à 25 / 50 / 100 m devient une ou plusieurs zones cohérentes, tout en gardant
les résolutions qui l'ont soutenue.

In [ ]:
consolidated_df = pd.DataFrame([
    {
        "feature": a.feature,
        "start_m": a.start_m,
        "end_m": a.end_m,
        "length_m": a.end_m - a.start_m,
        "resolutions_m": a.resolutions_m,
        "n_evidence": a.n_evidence,
        "multi_scale": a.multi_scale,
        "mean_confidence": round(a.mean_confidence, 3),
        "max_confidence": round(a.max_confidence, 3),
    }
    for a in consolidated_annotations
])

display(consolidated_df)

### Comparaison avant / après par caractéristique

In [ ]:
atomic_counts = (
    pd.Series([a.feature for a in atomic_annotations])
    .value_counts()
    .rename("atomic_count")
)

consolidated_counts = (
    pd.Series([a.feature for a in consolidated_annotations])
    .value_counts()
    .rename("consolidated_count")
)

comparison = (
    pd.concat([atomic_counts, consolidated_counts], axis=1)
    .fillna(0)
    .astype(int)
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(comparison)

## 5. Inspecter les preuves d'une zone consolidée

Cette cellule montre concrètement quelles fenêtres ont été fusionnées
pour former une zone.

In [ ]:
if consolidated_annotations:
    example = consolidated_annotations[0]

    print("Caractéristique :", example.feature)
    print("Zone :", example.start_m, "→", example.end_m, "m")
    print("Résolutions :", example.resolutions_m)
    print("Nombre de preuves :", example.n_evidence)
    print()

    display(pd.DataFrame([
        {
            "start_m": e.start_m,
            "end_m": e.end_m,
            "resolution_m": e.resolution_m,
            "confidence": e.confidence,
            "metrics": e.metrics,
        }
        for e in example.evidence
    ]))

## 6. Étape B — Combiner les caractéristiques différentes

Maintenant seulement, on construit les portions de l'itinéraire.

Exemple :

```text
pente_douce consolidée : 100────────────400
lacets consolidés      :       250────────────500
```

devient :

```text
100–250 → {pente_douce}
250–400 → {pente_douce, lacets}
400–500 → {lacets}
```

In [ ]:
def build_portions(consolidated):
    if not consolidated:
        return []

    boundaries = sorted({
        float(a.start_m)
        for a in consolidated
    } | {
        float(a.end_m)
        for a in consolidated
    })

    portions = []

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        if end <= start:
            continue

        active = [
            a for a in consolidated
            if a.start_m <= start and a.end_m >= end
        ]

        if not active:
            continue

        annotation_names = sorted({
            a.feature for a in active
        })

        details = [
            {
                "feature": a.feature,
                "source_start_m": a.start_m,
                "source_end_m": a.end_m,
                "resolutions_m": a.resolutions_m,
                "n_evidence": a.n_evidence,
                "multi_scale": a.multi_scale,
                "mean_confidence": a.mean_confidence,
                "max_confidence": a.max_confidence,
            }
            for a in active
        ]

        current = Portion(
            start_m=float(start),
            end_m=float(end),
            annotations=annotation_names,
            details=details,
        )

        # Si les mêmes caractéristiques continuent juste après,
        # on forme une portion plus longue.
        if (
            portions
            and portions[-1].end_m == current.start_m
            and portions[-1].annotations == current.annotations
        ):
            portions[-1].end_m = current.end_m

            # Les caractéristiques sont identiques.
            # On garde les détails déjà présents plutôt que les dupliquer.
        else:
            portions.append(current)

    return portions

In [ ]:
portions = build_portions(consolidated_annotations)

portions_df = pd.DataFrame([
    {
        "start_m": p.start_m,
        "end_m": p.end_m,
        "length_m": p.end_m - p.start_m,
        "annotations": ", ".join(p.annotations),
    }
    for p in portions
])

print("Nombre de portions finales :", len(portions))
display(portions_df)

## 7. Sauvegarder les résultats du notebook 04

On sauvegarde deux niveaux :

```text
agent2_annotations_consolidated.json
```

= une même caractéristique fusionnée à travers ses différentes échelles.

```text
agent2_portions.json
```

= portions finales avec combinaison de caractéristiques différentes.

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True, exist_ok=True)

consolidated_path = (
    output_dir / "agent2_annotations_consolidated.json"
)
portions_path = output_dir / "agent2_portions.json"
portions_csv_path = output_dir / "agent2_portions.csv"

with consolidated_path.open("w", encoding="utf-8") as f:
    json.dump(
        [a.model_dump() for a in consolidated_annotations],
        f,
        ensure_ascii=False,
        indent=2,
    )

with portions_path.open("w", encoding="utf-8") as f:
    json.dump(
        [p.model_dump() for p in portions],
        f,
        ensure_ascii=False,
        indent=2,
    )

portions_df.to_csv(portions_csv_path, index=False)

print("Résultats sauvegardés :")
print("-", consolidated_path)
print("-", portions_path)
print("-", portions_csv_path)

## 8. Ce que le notebook 05 recevra désormais

L'Agent 3 ne recevra plus les dizaines de fenêtres brutes.

Il recevra directement une portion comme :

```json
{
  "start_m": 250,
  "end_m": 400,
  "annotations": [
    "pente_douce",
    "lacets"
  ]
}
```

Les informations multi-échelles restent disponibles dans `details`,
mais la verbalisation peut rester simple.

### Séparation finale des responsabilités

```text
Notebook 03 / Agent 1
→ détecte et conserve les observations multi-échelles

Notebook 04 / Agent 2
→ consolide d'abord le même phénomène
→ combine ensuite les phénomènes différents

Notebook 05 / Agent 3
→ verbalise les portions finales
```